<audio controls>
  <source src="https://raw.githubusercontent.com/fair-vid/video-multimodal-pipeline/main/assets/audio/audio_extract_audio_from_video_tutorial.wav" type="audio/wav">
</audio>

# 🎧 Audio Extraction from Interview Videos  
*(This notebook can be executed end-to-end by selecting “Run all”)*
### Step 2 in the Agent-Based Digital Twin Demo

This notebook is the **second step** in the hands-on demonstration of the framework described in  
**“An Agent-Based Digital Twin for Transparent AI in Student Admissions.”**

In the real system, audio analysis runs **in parallel** with video and document agents.  
To keep the demo simple and reproducible in Colab, we execute these agents **one after another**, using separate notebooks.

---

## Where This Notebook Fits in the Workflow

- **Previous step**: Recording mock interview videos  
  👉 https://colab.research.google.com/drive/1mqcCrypaK3u_kvs7nSqX4XTvBgq4-shK

- **Current step (this notebook)**:  
  Extract audio (`.mp3`) from recorded interview videos.

- **Next steps** (linked at the end of this notebook):  
  Speech transcription, paralinguistic analysis, and fusion with other modalities.

---

## What This Notebook Does

This notebook scans your Google Drive for recorded interview videos and extracts the audio track from each video file.

Specifically, it:
- searches for video files in `video_interviews/`,
- preserves the application folder structure,
- creates corresponding audio files in `video_interviews_audio_files/`,
- skips files that were already processed.

The extracted audio files are used later by AI agents responsible for speech recognition and behavioral analysis.

---

## Important Notes

- All processing happens **locally inside your Colab session**
- Audio files are saved **only to your Google Drive**
- No data is uploaded to external services
- You can re-run this notebook safely; existing audio files will not be overwritten

Run the cells below in order to continue the pipeline.
Links to the next notebooks are provided at the end.


In [ ]:
import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive is already mounted")

Mounting Google Drive...
Mounted at /content/drive


### Select target folders

If the ***FILTERED_LIST_Applicant_Application_ids_OneApp.tsv*** filter file exists, only the folders listed there are processed. If it is missing, the notebook falls back to all subdirectories in the application root.

In [ ]:
'''import csv
from pathlib import Path

drive_dir = Path("/content/drive/MyDrive/colab-output/dream_applicant_application/")
filter_file = drive_dir / "FILTERED_LIST_Applicant_Application_ids_OneApp.tsv"

allowed = set()
if filter_file.exists():
    with open(filter_file, "r", encoding="utf-8") as f:
        for row in csv.reader(f, delimiter="\t"):
            if row and row[0].strip():
                allowed.add(row[0].strip())

    subdirs = [p for p in drive_dir.iterdir() if p.is_dir() and p.name in allowed]
    print("Using filtered directories:", [d.name for d in subdirs])
else:
    subdirs = [p for p in drive_dir.iterdir() if p.is_dir()]
    print("Filter file missing, using all directories:", [d.name for d in subdirs])

 '''

Using filtered directories: ['60811_46042']


In [ ]:
import pandas as pd
from pathlib import Path

drive_dir = Path("/content/drive/MyDrive/colab-output/dream_applicant_application/")
filter_file = drive_dir / "FILTERED_LIST_Applicant_Application_ids_OneApp.tsv"

allowed = set()

if filter_file.exists():
    df = pd.read_csv(filter_file, sep='\t')
    if not df.empty:
        allowed = set(df.iloc[:, 0].astype(str).str.strip())  # Take the first column, convert to string, strip spaces, and make a set
    if allowed:
        subdirs = [p for p in drive_dir.iterdir() if p.is_dir() and p.name in allowed]
        print("Using filtered directories:", [d.name for d in subdirs])
    else:
        subdirs = [p for p in drive_dir.iterdir() if p.is_dir()]
        print("Filter file is empty, using all directories")
else:
    subdirs = [p for p in drive_dir.iterdir() if p.is_dir()]
    print("Filter file missing, using all directories")

Using filtered directories: ['60811_46042']


In [ ]:
from pathlib import Path

drive_dir = "/content/drive/MyDrive/colab-output/dream_applicant_application/"

# Dictionary to store files by subdirectory
files_by_subdir = {}

# Iterate through direct subdirectories
#for subdir in Path(drive_dir).iterdir():
for subdir in subdirs:
    if subdir.is_dir():
        # Check if 'video_interviews' exists in this subdirectory
        documents_video_path = subdir / "video_interviews"

        if documents_video_path.exists() and documents_video_path.is_dir():
            # Store both direct files and files in subdirectories
            video_info = {
                'direct_files': [],
                'subdirs': {}
            }

            # Iterate through contents of video_interviews
            for item in documents_video_path.iterdir():
                if item.is_file():
                    # Direct video file
                    video_info['direct_files'].append(item)
                elif item.is_dir():
                    # Subdirectory with video files
                    subdir_files = [f for f in item.iterdir() if f.is_file()]
                    if subdir_files:
                        video_info['subdirs'][item.name] = subdir_files

            files_by_subdir[subdir.name] = video_info

            total_files = len(video_info['direct_files']) + sum(len(files) for files in video_info['subdirs'].values())
            print(f"Found {total_files} files in {subdir.name}/video_interviews/")

# Now you can use the files later
for subdir_name, video_info in files_by_subdir.items():
    print(f"\n{subdir_name}:")

    # Print direct files
    if video_info['direct_files']:
        print("  Direct files:")
        for file in video_info['direct_files']:
            print(f"    - {file.name}")

    # Print files in subdirectories
    if video_info['subdirs']:
        print("  Files in subdirectories:")
        for subdir_name, files in video_info['subdirs'].items():
            print(f"    {subdir_name}/")
            for file in files:
                print(f"      - {file.name}")

Found 7 files in 60811_46042/video_interviews/

60811_46042:
  Direct files:
    - 51375_What_new_hobbies_or_activities_would_you_like_to_t.mp4
    - 51376_Which_qualities_of_VILNIUS_TECH_stood_out_to_you_a.mp4
    - 51377_How_would_your_friends_characterize_you_.mp4
    - 51378_How_do_you_deal_with_unexpected_changes_or_disrupt.mp4
    - 51379_What_does_innovation_mean_to_you_personally__how_d.mp4
    - 51380_What_methods_do_you_use_to_make_sure_you_stay_on_t.mp4
    - 51381_What_made_you_choose_this_specific_programme_at_VI.mp4


In [ ]:
import os
from moviepy.editor import VideoFileClip

# Extract audio from video files
for subdir_name, video_info in files_by_subdir.items():
    print(f"\nProcessing {subdir_name}...")

    # Create audio directory path
    audio_base_dir = Path(drive_dir) / subdir_name / "video_interviews_audio_files"

    has_audio_files = False

    # Process direct video files
    if video_info['direct_files']:
        for video_file in video_info['direct_files']:
            # Prepare audio file path (directly in audio base dir)
            audio_filename = video_file.stem + ".mp3"
            audio_path = audio_base_dir / audio_filename

            # Check if audio file already exists
            if audio_path.exists():
                print(f"  ⏭️  Skipping {video_file.name} (audio already exists)")
                has_audio_files = True
                continue

            # Create audio directory if needed
            if not has_audio_files:
                os.makedirs(audio_base_dir, exist_ok=True)
                has_audio_files = True

            # Extract audio
            try:
                print(f"  🎵 Extracting audio from {video_file.name}")
                video = VideoFileClip(str(video_file))
                video.audio.write_audiofile(str(audio_path), verbose=False, logger=None)
                video.close()
            except Exception as e:
                print(f"  ❌ Error processing {video_file.name}: {e}")

    # Process files in subdirectories
    if video_info['subdirs']:
        for subdir_folder_name, files in video_info['subdirs'].items():
            # Create corresponding subdirectory in audio folder
            audio_subdir = audio_base_dir / subdir_folder_name

            for video_file in files:
                # Prepare audio file path
                audio_filename = video_file.stem + ".mp3"
                audio_path = audio_subdir / audio_filename

                # Check if audio file already exists
                if audio_path.exists():
                    print(f"  ⏭️  Skipping {subdir_folder_name}/{video_file.name} (audio already exists)")
                    has_audio_files = True
                    continue

                # Create audio subdirectory if needed
                if not has_audio_files:
                    os.makedirs(audio_base_dir, exist_ok=True)
                    has_audio_files = True
                os.makedirs(audio_subdir, exist_ok=True)

                # Extract audio
                try:
                    print(f"  🎵 Extracting audio from {subdir_folder_name}/{video_file.name}")
                    video = VideoFileClip(str(video_file))
                    video.audio.write_audiofile(str(audio_path), verbose=False, logger=None)
                    video.close()
                except Exception as e:
                    print(f"  ❌ Error processing {subdir_folder_name}/{video_file.name}: {e}")

print("\n✅ All audio files extracted!")


Processing 60811_46042...
  🎵 Extracting audio from 51375_What_new_hobbies_or_activities_would_you_like_to_t.mp4
  🎵 Extracting audio from 51376_Which_qualities_of_VILNIUS_TECH_stood_out_to_you_a.mp4
  🎵 Extracting audio from 51377_How_would_your_friends_characterize_you_.mp4
  🎵 Extracting audio from 51378_How_do_you_deal_with_unexpected_changes_or_disrupt.mp4
  🎵 Extracting audio from 51379_What_does_innovation_mean_to_you_personally__how_d.mp4
  🎵 Extracting audio from 51380_What_methods_do_you_use_to_make_sure_you_stay_on_t.mp4
  🎵 Extracting audio from 51381_What_made_you_choose_this_specific_programme_at_VI.mp4

✅ All audio files extracted!



---

## ▶️ Next Steps: Parallel Audio Analysis Agents

At this stage, the interview videos have been converted into structured audio files.
In the process digital twin described in the paper, **multiple audio-level agents can operate in parallel**, each focusing on a different analytical dimension.

### Primary path: Speech Transcription Agent

One downstream agent performs **speech transcription**, transforming spoken responses into inspectable text artifacts for later linguistic and semantic analysis.

**Step 3 – Speech Transcription from Interview Audio**
👉 [https://colab.research.google.com/drive/1tXdu3AAVjaYeGjpG8k5sA4Dk2SB9Z-u3](https://colab.research.google.com/drive/1tXdu3AAVjaYeGjpG8k5sA4Dk2SB9Z-u3)

This notebook:

* automatically detects extracted audio files,
* applies a local speech-to-text model,
* stores plain-text transcripts alongside the application data,
* preserves folder structure for trace-level inspection.

The resulting transcripts are consumed by language and authenticity analysis agents in later steps.

---
---

### Alternative / parallel path: Audio Emotion & Prosody Agent

In parallel to transcription, the same audio files can be analyzed for **emotional and prosodic characteristics**, capturing aspects of delivery that are not visible in text alone.

**Audio Emotion & Prosody Analysis**
👉 [https://colab.research.google.com/drive/1fcOBNAjy_B7p3Kh0TOKgiHhwlrALUFx9](https://colab.research.google.com/drive/1fcOBNAjy_B7p3Kh0TOKgiHhwlrALUFx9)

This agent extracts:

* speech tempo and pause patterns,
* pitch statistics and variability,
* probabilistic emotion signals from vocal expression.

These features complement transcript-based evaluation and are later fused with video and language-level evidence inside the digital twin.

> **Note:**
> In the Colab demo, these agents are executed as separate notebooks for clarity and reproducibility.
> In a production system, they would run concurrently as part of a coordinated agent ensemble.

> **Repository notice:**
> Links to the complete GitHub repository and orchestration code will be added after the anonymous review process is completed.

---









